# Hands-on Session 3: Evaluation and Interpretation of Spatial Protein Predictions

**Time:** 11:35-12:00  
**Lead:** Zeyuan

Goals:
- Evaluate inferred proteins with correlation metrics.
- Quantify spatial coherence with Moran's I.
- Compare transcript and inferred protein landscapes.
- Discuss biological interpretation and common pitfalls.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from dgat_tutorial.data import find_dgat_h5ad, load_tutorial_data
from dgat_tutorial.dgat import run_demo_dgat_inference
from dgat_tutorial.evaluation import morans_i, protein_correlations
from dgat_tutorial.plotting import plot_correlation_bar, plot_spatial_feature

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = repo_root / "data" / "raw"
processed_dir = repo_root / "data" / "processed"
fig_dir = repo_root / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

## Load Observed and Predicted Proteins

In [ ]:
dgat_h5ad = find_dgat_h5ad(data_dir)
dataset = load_tutorial_data(data_dir, allow_demo=True)
print(f"Loaded DGAT AnnData file: {dgat_h5ad}" if dgat_h5ad else "Loaded synthetic fallback data")

spots = dataset.spots
transcripts = dataset.transcripts
proteins = dataset.proteins

prediction_path = processed_dir / "predicted_proteins.csv"
if prediction_path.exists():
    predicted_proteins = pd.read_csv(prediction_path, index_col=0)
else:
    predicted_proteins = run_demo_dgat_inference(transcripts, proteins)

common_spots = spots.index.intersection(proteins.index).intersection(predicted_proteins.index)
spots = spots.loc[common_spots]
proteins = proteins.loc[common_spots]
predicted_proteins = predicted_proteins.loc[common_spots]

print(f"Evaluating {len(common_spots)} spots and {predicted_proteins.shape[1]} proteins")

## Correlation-Based Evaluation

In [ ]:
correlations = protein_correlations(proteins, predicted_proteins)
display(correlations)

plot_correlation_bar(correlations, metric="pearson")
plt.tight_layout()
plt.savefig(fig_dir / "session03_prediction_correlations.png", dpi=160)
plt.show()

## Spatial Coherence with Moran's I

In [ ]:
moran_rows = []
for protein in predicted_proteins.columns:
    moran_rows.append(
        {
            "protein": protein,
            "observed_morans_i": morans_i(proteins[protein], spots),
            "predicted_morans_i": morans_i(predicted_proteins[protein], spots),
        }
    )

moran_table = pd.DataFrame(moran_rows)
display(moran_table)

sns.scatterplot(data=moran_table, x="observed_morans_i", y="predicted_morans_i", hue="protein", s=80)
plt.axline((0, 0), slope=1, color="black", linewidth=0.8)
plt.title("Observed vs predicted spatial coherence")
plt.tight_layout()
plt.savefig(fig_dir / "session03_morans_i.png", dpi=160)
plt.show()

## Compare Transcript and Inferred Protein Landscapes

In [ ]:
protein = correlations.iloc[0]["protein"]
gene = transcripts.columns[0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
plot_spatial_feature(spots, transcripts[gene], f"Transcript {gene}", cmap="magma", ax=axes[0])
plot_spatial_feature(spots, proteins[protein], f"Observed {protein}", cmap="viridis", ax=axes[1])
plot_spatial_feature(spots, predicted_proteins[protein], f"Predicted {protein}", cmap="viridis", ax=axes[2])
plt.tight_layout()
plt.savefig(fig_dir / "session03_landscape_comparison.png", dpi=160)
plt.show()

## Interpretation Prompts

- Which proteins are predicted well by correlation but less well by spatial coherence?
- Which proteins show plausible spatial structure despite moderate pointwise correlation?
- Where might transcript-protein discordance be biological rather than model error?
- What batch, antibody, tissue-boundary, or cell-composition effects could mislead evaluation?